# Variabilidade, intervalos e testes de hipótese

[▶ Abrir este notebook no Google Colab](https://colab.research.google.com/github/correa-ufrrj/disciplina_computacao_aplicada_humanidades_digitais/blob/main/unidade_05/02_incerteza_e_testes_de_hipotese_revisado.ipynb)

No Notebook 01, estimamos diferenças observadas entre grupos e discutimos tamanho de efeito, sensibilidade e relevância substantiva. Agora acrescentaremos outra dimensão: **a incerteza**.

A pergunta central deste notebook é:

> **Que variação esperaríamos para nossa estimativa se o procedimento de obtenção dos dados pudesse ser repetido?**

Essa pergunta não é a mesma que perguntar se a diferença observada é grande, importante ou historicamente significativa. Também não corrige problemas de seleção, cobertura ou mensuração.

## 1. Antes da inferência: de onde vêm os dados?

Chamamos de **variabilidade amostral** a variação que uma estatística apresentaria entre amostras obtidas pelo mesmo procedimento.

Ela não deve ser confundida com erro de medição, erro de transcrição ou codificação, lacunas de cobertura, viés de seleção, mudanças históricas reais ou escolhas de operacionalização.

| Situação | O intervalo amostral resolve? | Razão |
|---|---|---|
| amostra probabilística de uma população definida | pode quantificar parte da incerteza | o desenho sustenta a generalização |
| corpus completo para a pergunta | geralmente não é a questão central | a diferença do corpus já foi observada |
| coleção de conveniência | não corrige a seleção | a reamostragem herda a composição disponível |

Neste material, os dados são **sintéticos e didáticos**. Trataremos o mecanismo de geração como se definisse um procedimento repetível apenas para compreender os métodos. Não faremos inferências históricas.

In [ ]:
# @title Preparação do ambiente — execute esta célula no Google Colab
from pathlib import Path
import os
import subprocess

URL_REPOSITORIO = "https://github.com/correa-ufrrj/disciplina_computacao_aplicada_humanidades_digitais.git"
REPOSITORIO = Path("/dados/disciplina_computacao_aplicada_humanidades_digitais")
PASTA_UNIDADE = REPOSITORIO / "unidade_05"

try:
    import google.colab  # type: ignore
    EM_COLAB = True
except ImportError:
    EM_COLAB = False

if EM_COLAB:
    if not (REPOSITORIO / ".git").exists():
        subprocess.run(
            ["git", "clone", "--depth", "1", "--branch", "main",
             URL_REPOSITORIO, str(REPOSITORIO)],
            check=True,
        )
    os.chdir(PASTA_UNIDADE)
    print("Ambiente preparado em:", Path.cwd())
else:
    print("Ambiente local: nenhuma clonagem necessária.")

Usaremos o mesmo conjunto ampliado da versão revisada da Unidade 5, com **120 documentos fictícios**.

Se `dados/documentos.csv` já existir, ele será carregado. Caso contrário, o notebook reproduzirá o mesmo mecanismo gerador usado no guia e no Notebook 01.

In [ ]:
import math
from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from IPython.display import display

ARQUIVO = Path("dados/documentos.csv")

def gerar_dados():
    SEED, N = 20261058, 120
    rng = np.random.default_rng(SEED)
    linhas = []
    for ano in range(1890, 1910):
        t = (ano - 1890) / 19
        p_capital = 0.12 + 0.76 * t
        n_capital = int(np.clip(rng.binomial(6, p_capital), 1, 5))
        locais_ano = np.array(["Capital"] * n_capital + ["Interior"] * (6 - n_capital))
        rng.shuffle(locais_ano)
        linhas.extend((ano, local) for local in locais_ano)

    rng.shuffle(linhas)
    anos = np.array([x[0] for x in linhas])
    locais = np.array([x[1] for x in linhas])
    capital = locais == "Capital"

    generos = rng.choice(["notícia", "editorial", "carta"], N, p=[0.45, 0.30, 0.25])
    temas_possiveis = np.array(["educação", "trabalho", "progresso", "saúde"])
    probs = {
        "notícia": [0.20, 0.25, 0.15, 0.40],
        "editorial": [0.20, 0.30, 0.35, 0.15],
        "carta": [0.40, 0.25, 0.20, 0.15],
    }
    temas = np.array([rng.choice(temas_possiveis, p=probs[g]) for g in generos])

    efeito_genero = {"notícia": 0.03, "editorial": 0.24, "carta": -0.18}
    log_palavras = (
        math.log(650)
        + np.array([efeito_genero[g] for g in generos])
        + np.where(capital, 0.64, 0.0)
        - 0.016 * (anos - 1890)
        + rng.normal(0, 0.24, N)
    )
    palavras = np.maximum(140, np.rint(np.exp(log_palavras))).astype(int)

    especiais = rng.choice(N, 3, replace=False)
    palavras[especiais] = np.rint(
        palavras[especiais] * rng.uniform(1.65, 2.05, 3)
    ).astype(int)

    densidade = np.clip(rng.normal(365, 38, N), 270, 470)
    paginas = np.maximum(1, np.rint(palavras / densidade)).astype(int)

    taxa = {"notícia": 7.0, "editorial": 4.7, "carta": 5.8}
    pessoas = rng.poisson(palavras / 1000 * np.array([taxa[g] for g in generos]))

    df = pd.DataFrame({
        "id_documento": [f"R{i:03d}" for i in range(1, N + 1)],
        "ano": anos, "genero": generos, "local": locais, "tema": temas,
        "palavras": palavras, "paginas": paginas, "pessoas": pessoas,
    })
    ARQUIVO.parent.mkdir(exist_ok=True)
    df.to_csv(ARQUIVO, index=False)
    return df

if ARQUIVO.exists():
    dados = pd.read_csv(ARQUIVO)
    print("Dados carregados de:", ARQUIVO)
else:
    dados = gerar_dados()
    print("Dados reproduzidos em:", ARQUIVO)

print("Dimensões:", dados.shape)
display(dados.head())

## 2. *Bootstrap* e distribuição da estimativa

No Notebook 01, a estimativa principal foi a diferença entre as médias de `palavras` dos grupos `Capital` e `Interior`.

$$
\Delta_{\mathrm{obs}}
=
\bar{x}_{Capital}
-
\bar{x}_{Interior}.
$$

O *bootstrap* procura responder: se pudéssemos repetir um procedimento semelhante ao que produziu os dados observados, quanto essa diferença tenderia a variar?

Em cada repetição $b$, reamostramos com reposição dentro de cada grupo e calculamos:

$$
\Delta_b
=
\bar{x}_{Capital,b}
-
\bar{x}_{Interior,b}.
$$

A coleção desses valores forma uma **distribuição bootstrap da estimativa**. Ela não representa novos passados possíveis; representa a variação produzida pela regra de reamostragem sobre os dados observados.

In [ ]:
a = dados.loc[dados["local"].eq("Capital"), "palavras"].to_numpy()
b = dados.loc[dados["local"].eq("Interior"), "palavras"].to_numpy()

diferenca_observada = a.mean() - b.mean()

rng = np.random.default_rng(20260906)
B = 4000
diferencas_bootstrap = np.array([
    rng.choice(a, size=len(a), replace=True).mean()
    - rng.choice(b, size=len(b), replace=True).mean()
    for _ in range(B)
])

print(f"Diferença observada: {diferenca_observada:.2f} palavras")
pd.Series(diferencas_bootstrap).describe(percentiles=[0.025, 0.5, 0.975])

In [ ]:
plt.figure(figsize=(8, 5))
plt.hist(diferencas_bootstrap, bins=35)
plt.axvline(diferenca_observada, linestyle="--", label="diferença observada")
plt.xlabel("Diferença de médias: Capital - Interior")
plt.ylabel("Frequência")
plt.title("Distribuição bootstrap da diferença")
plt.legend()
plt.show()

A largura da distribuição indica quanto a estimativa varia sob a regra de reamostragem adotada. Uma distribuição mais estreita sugere maior precisão **dentro desse procedimento**; isso não significa, por si só, maior validade histórica do corpus.

## 3. Intervalo de confiança

Usaremos o intervalo percentil introdutório:

$$
IC_{95\%}
=
\left[
Q_{0{,}025}(\Delta_b),
Q_{0{,}975}(\Delta_b)
\right].
$$

Em repetições do procedimento, cerca de 95% dos intervalos construídos dessa maneira cobririam o parâmetro-alvo sob as condições do método.

Depois que um intervalo específico foi calculado, não dizemos que há 95% de probabilidade frequentista de o parâmetro fixo estar nele. O intervalo também **não contém 95% dos documentos**.

In [ ]:
limite_inferior, limite_superior = np.quantile(
    diferencas_bootstrap, [0.025, 0.975]
)

tabela_intervalo = pd.DataFrame([{
    "estimativa": diferenca_observada,
    "limite inferior": limite_inferior,
    "limite superior": limite_superior,
    "unidade": "palavras por documento",
}])

display(tabela_intervalo.round(2))

O intervalo deve ser lido junto com a estimativa pontual, a unidade da variável, o desenho de obtenção dos dados e as fontes de incerteza que o método não representa.

Um intervalo estreito em uma coleção enviesada continua sendo um intervalo estreito sobre uma coleção enviesada.

## 4. Hipótese nula e teste de permutação

O *bootstrap* foi usado para estudar a variabilidade da estimativa. O teste de permutação responde a outra pergunta: quão incompatível é o resultado observado com uma **hipótese nula específica**?

A hipótese nula didática será:

> sob o modelo nulo, os rótulos `Capital` e `Interior` são intercambiáveis em relação à variável `palavras`.

Mantemos valores e tamanhos dos grupos, embaralhamos os rótulos e recalculamos a diferença.

Para $M$ permutações:

$$
p
=
\frac{
1+\sum_{m=1}^{M}
\mathbf{1}\left(|T_m|\geq|T_{obs}|\right)
}{
M+1
}.
$$

Usamos valores absolutos porque o teste é bilateral: resultados extremos nas duas direções contam contra o modelo nulo.

In [ ]:
rng = np.random.default_rng(20260906)
combinados = np.concatenate([a, b])
M = 5000
diferencas_nulas = np.empty(M)

for m in range(M):
    embaralhados = rng.permutation(combinados)
    diferencas_nulas[m] = (
        embaralhados[:len(a)].mean()
        - embaralhados[len(a):].mean()
    )

valor_p = (
    1 + np.sum(np.abs(diferencas_nulas) >= abs(diferenca_observada))
) / (M + 1)

display(pd.Series({
    "diferença observada": diferenca_observada,
    "valor de p bilateral": valor_p,
}).round(4))

In [ ]:
plt.figure(figsize=(8, 5))
plt.hist(diferencas_nulas, bins=35)
plt.axvline(diferenca_observada, linestyle="--", label="diferença observada")
plt.axvline(-diferenca_observada, linestyle=":", label="valor simétrico")
plt.xlabel("Diferença de médias sob permutação")
plt.ylabel("Frequência")
plt.title("Distribuição nula por permutação")
plt.legend()
plt.show()

## 5. Como interpretar o valor de *p*

O valor de *p* responde a uma pergunta condicional:

> **Se a hipótese nula e o procedimento especificado fossem adequados, quão extremos seriam resultados como o observado?**

Ele **não** informa:

- a probabilidade de a hipótese nula ser verdadeira;
- a probabilidade de a hipótese alternativa ser verdadeira;
- o tamanho da diferença;
- a importância substantiva da diferença;
- a probabilidade de replicação;
- a ausência de viés;
- a qualidade do corpus;
- a qualidade da operacionalização.

Por isso, um valor pequeno de *p* não substitui a estimativa, o tamanho de efeito, o intervalo ou a interpretação substantiva. Também não devemos transformar automaticamente `0,05` em uma fronteira entre “verdadeiro” e “falso”.

## 6. Significância estatística e relevância substantiva

Uma diferença muito pequena pode apresentar valor de *p* baixo quando há muitos dados e pouca variabilidade. Uma diferença substantivamente importante pode permanecer imprecisa quando o número de observações é pequeno ou os dados são muito heterogêneos.

Portanto:

$$
\text{valor de }p
\neq
\text{tamanho de efeito}
\neq
\text{relevância substantiva}.
$$

O valor de *p* também não corrige o processo de seleção dos dados. Se o corpus é de conveniência, a permutação continua operando sobre essa mesma coleção.

## 7. Reunindo as peças

A análise quantitativa possui componentes diferentes:

1. **estimativa** — qual diferença observamos?;
2. **tamanho de efeito** — qual a magnitude da diferença em relação à variabilidade?;
3. **incerteza** — quanto a estimativa varia sob um procedimento de reamostragem?;
4. **teste de hipótese** — quão incompatível é o resultado com um modelo nulo?;
5. **relevância substantiva** — o que a diferença significa para a pergunta de pesquisa?

Nenhuma dessas dimensões substitui as demais.

### U05-A03 — Atividade integrada — ficha inferencial

**Modalidade:** dupla. **Tempo:** 30 minutos.

1. declare a população-alvo ou o universo ao qual a análise pretende se referir;
2. descreva o procedimento de seleção ou formação do corpus;
3. informe a estimativa observada e sua unidade;
4. apresente o intervalo bootstrap;
5. recupere o tamanho de efeito calculado no Notebook 01;
6. formule explicitamente a hipótese nula do teste de permutação;
7. interprete o valor de *p* sem transformá-lo em uma decisão binária;
8. indique pelo menos duas fontes de incerteza que o intervalo ou o teste não representam;
9. escreva uma conclusão substantiva limitada.

**População-alvo ou universo:** Escreva aqui.

**Procedimento de seleção:** Escreva aqui.

**Estimativa e intervalo:** Escreva aqui.

**Tamanho de efeito:** Escreva aqui.

**Hipótese nula:** Escreva aqui.

**Interpretação do valor de p:** Escreva aqui.

**Fontes de incerteza não representadas:** Escreva aqui.

**Conclusão substantiva e limites:** Escreva aqui.

O eixo quantitativo está completo. No Notebook 03, a comparação muda de objeto: representaremos documentos como termos e pesos antes de calcular semelhanças.